In [2]:
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import (
    ChatPromptTemplate,PromptTemplate,
    SystemMessagePromptTemplate,AIMessagePromptTemplate,
    HumanMessagePromptTemplate, FewShotChatMessagePromptTemplate
)
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain






import gradio as gr
import warnings

from config import get_llm   # must return a LangChain-compatible LLM

warnings.filterwarnings("ignore") # Backup embeddings


In [3]:
loader = PyPDFLoader("aiayn.pdf")
docs = loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=150,
    chunk_overlap=20
)
chunks = splitter.split_documents(docs)

In [5]:
vectordb = Chroma.from_documents(
        chunks,
        HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))
retriever = vectordb.as_retriever(search_kwargs={"k": 5})


In [ ]:
# from transformers import AutoConfig, AutoTokenizer

# model_name = "sentence-transformers/all-MiniLM-L6-v2"

# config = AutoConfig.from_pretrained(model_name)
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# print("Max position embeddings:", config.max_position_embeddings)
# print("Tokenizer model max length:", tokenizer.model_max_length)

Max position embeddings: 512
Tokenizer model max length: 512


In [ ]:
# vectordb.delete_collection()

In [6]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [7]:
memory.clear()


In [7]:
llm=ChatOllama(model="llama3", temperature=0.2)

In [ ]:
# prompt_template = """Use the information from the document to answer. Be sarcastic

# {context}

# Question: {question}
# """

# PROMPT = PromptTemplate(
#     template=prompt_template, input_variables=["context", "question"]
# )

# chain_type_kwargs = {"prompt": PROMPT}

In [14]:
prompt_template = """Use the following to answer.

Chat history:
{chat_history}

retreived docs:
{retreived_docs}

Question: {question}
Answer:
"""

PROMPT_CHAT = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("""You are a helpful assistant. Answer the question using the conversation history and retrieved documents.
IMPORTANT INSTRUCTIONS:
1. First check if the question was already answered in the Conversation History
2. If the answer is in the Conversation History, use that answer
3. Only use Retrieved Documents if they are relevant to the question
4. If Retrieved Documents discuss unrelated topics, IGNORE them
5. Give a clear, direct answer without mentioning irrelevant information
6. If you dont find any answer if in conversation history or reterived document then say 'I dont know' """),
    HumanMessagePromptTemplate.from_template(prompt_template)
])

In [15]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

qa_3 = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    chain_type="stuff",
    combine_docs_chain_kwargs= {"prompt": PROMPT_CHAT,
    "document_variable_name": "retreived_docs"
},
    return_source_documents=True
)

In [ ]:
#memory.save_context({"question": 'What is vacation policy'}, {"answer": 'Employees get 15 days of paid vacation per year.'})

In [36]:
from langchain_classic.schema import HumanMessage, AIMessage, SystemMessage


In [ ]:
#memory.clear()

In [ ]:
query = "what is not a transformer"

result = qa_3.invoke({"question": query})

print("Answer:", result["answer"])

Answer: I can answer that!

According to the retrieved documents, a Transformer is a type of transduction model. It allows for significant parallelization and has reached a new state of the art in certain areas.


### Evaluation

In [19]:
from typing_extensions import Annotated, TypedDict

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

# Grade prompt
correctness_instructions = """You are a teacher grading a quiz.

You will be given:
- a QUESTION
- a GROUND TRUTH ANSWER
- a STUDENT ANSWER

Grade criteria:
1. Judge factual accuracy ONLY relative to the ground truth.
2. Ignore extra correct details unless they contradict the ground truth.
3. Mark False only if there are factual errors or contradictions.

For the explanation:
- Always provide 2 to 4 concise bullet points.
- Each bullet must reference a concrete comparison between the student answer and the ground truth.
- Do NOT leave the explanation empty.
- Do NOT use generic placeholders like "Correct" or "Step-by-Step Reasoning" """

# Grader LLM
grader_llm = llm.with_structured_output(
    CorrectnessGrade, method="json_schema", strict=True
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
    QUESTION: {inputs['question']}
    GROUND TRUTH ANSWER: {reference_outputs['answer']}
    STUDENT ANSWER: {outputs['answer']}"""
    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])
    return grade

In [20]:
result = qa_3.invoke({"question": 'what is transformer'})

is_correct = correctness(
    inputs={"question": result['question']},
    outputs={"answer": result['answer']
},
    reference_outputs={
        "answer": "Transformer is a type of transduction model that allows for significant parallelization and can reach new state-of-the-art results"
    }
)




In [24]:
result['answer']

'According to the chat history, the answer is: The Transformer is a model that enables parallelization and achieves a new state of the art in certain applications.'

In [23]:
is_correct['explanation']

"The student's answer is mostly accurate, but not entirely factual. Here are some key points for comparison:\n\n• The student correctly mentioned that the Transformer allows for significant parallelization, which aligns with the ground truth.\n• However, the student did not explicitly state that it's a type of transduction model, which is an important detail in the ground truth.\n• Additionally, while the student mentioned achieving new state-of-the-art results, they didn't specify 'in certain applications', which is present in the ground truth. Overall, the student's answer is mostly correct but lacks some crucial details.\n\nGrade: 3/4"